In [1]:
%load_ext autoreload
%autoreload 2

# Extract Information from Pdf Acta de Votacion

In [3]:
from pathlib import Path
from elecc_colombia.config import setup_logger
from elecc_colombia.config import (RAW_DATA_DIR, 
                                   INTERIM_DATA_DIR,
                                   PROCESSED_DATA_DIR)
from elecc_colombia.acta_handwrite_ocr_reader import (read_nivelacion,
                                                      read_digit)

setup_logger()

2026-06-16 15:59:47.633 | INFO     | elecc_colombia.config:<module>:12 - PROJ_ROOT path is: /Users/mmh54/Documents/github/escrutinio-elecciones-col


<loguru.logger handlers=[(id=2, level=10, sink=stderr)]>

In [3]:
pdf_path = RAW_DATA_DIR / "E14_XXX_X_72_006_000_00_000_X_XXX.pdf"
debug_dir = INTERIM_DATA_DIR / "debug_acta"
for label, value in read_nivelacion(pdf_path, debug_dir=debug_dir).items():
    print(f"  {label}: {value}")

Using CPU. Note: This module is much faster with a GPU.


  [debug] saved full page → /Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/page1_full.png


/Users/mmh54/Documents/github/escrutinio-elecciones-col/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  [debug] matched 'TOTAL VOTANTES FORMULARIO E-11' at y=2775–2912
  [debug] matched 'TOTAL VOTOS EN LA URNA' at y=3069–3205
  [debug] matched 'TOTAL VOTOS INCINERADOS' at y=3374–3496
  [debug] saved crop → /Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/crop_total_votantes_formulario_e-11.png
  [debug] box 0 → '4'
  [debug] box 1 → '0'
  [debug] box 2 → ''
  [debug] saved crop → /Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/crop_total_votos_en_la_urna.png
  [debug] box 0 → '4'
  [debug] box 1 → ''
  [debug] box 2 → ''
  [debug] saved crop → /Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/crop_total_votos_incinerados.png
  [debug] box 0 → ''
  [debug] box 1 → ''
  [debug] box 2 → ''
  TOTAL VOTANTES FORMULARIO E-11: 400
  TOTAL VOTOS EN LA URNA: 400
  TOTAL VOTOS INCINERADOS: None


In [5]:
import easyocr
import re

reader = easyocr.Reader(['en'])  # downloads model on first run

def read_handwritten_number(image_path: str) -> int | None:
    results = reader.readtext(image_path, detail=0, allowlist='0123456789')
    
    for text in results:
        digits = re.sub(r'\D', '', text)
        if 1 <= len(digits) <= 3:
            return int(digits)
    return None

In [9]:
from PIL import Image
image_pathname = "/Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/crop_total_votos_en_la_urna_box1.png"

img = Image.open(image_pathname)
number = read_digit(img)

print(f"Extracted number: {number}")

Extracted number: None


In [7]:
#show image
img.show()

In [8]:
image_pathname = "/Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/crop_total_votos_en_la_urna_box1.png"
number = read_handwritten_number(image_pathname)
print(f"Extracted number: {number}")

Extracted number: None


## Read Images with numbers

In [12]:
    #pdf_path = RAW_DATA_DIR / "E14_XXX_X_72_006_000_00_000_X_XXX.pdf"
    
    pdf_path = RAW_DATA_DIR / "E14_XXX_X_72_008_000_00_000_X_XXX (1).pdf"
    debug_dir = INTERIM_DATA_DIR / "debug_acta"
    for label, value in read_nivelacion(pdf_path, debug_dir=debug_dir).items():
        print(f"  {label}: {value}")

Using CPU. Note: This module is much faster with a GPU.


  [debug] saved full page → /Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/page1_full.png


/Users/mmh54/Documents/github/escrutinio-elecciones-col/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  [debug] matched 'TOTAL VOTANTES FORMULARIO E-11' at y=2755–2866
  [debug] matched 'TOTAL VOTOS EN LA URNA' at y=3050–3161
  [debug] matched 'TOTAL VOTOS INCINERADOS' at y=3335–3457
  [debug] saved crop → /Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/crop_total_votantes_formulario_e-11.png
  [debug] box 0 → ''
  [debug] box 1 → '2'
  [debug] box 2 → ''
  [debug] saved crop → /Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/crop_total_votos_en_la_urna.png
  [debug] box 0 → ''
  [debug] box 1 → '2'
  [debug] box 2 → '0'
  [debug] saved crop → /Users/mmh54/Documents/github/escrutinio-elecciones-col/data/interim/debug_acta/crop_total_votos_incinerados.png
  [debug] box 0 → ''
  [debug] box 1 → '-'
  [debug] box 2 → ''
  TOTAL VOTANTES FORMULARIO E-11: 20
  TOTAL VOTOS EN LA URNA: 20
  TOTAL VOTOS INCINERADOS: None


## Tensor Flow Approach

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

model = load_model("digit_model.keras")

img = cv2.imread("number.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

blur = cv2.GaussianBlur(gray, (5, 5), 0)
thresh = cv2.threshold(
    blur, 0, 255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)[1]

contours, _ = cv2.findContours(
    thresh,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

digits = []

for c in contours:
    x, y, w, h = cv2.boundingRect(c)

    if w < 5 or h < 10:
        continue

    roi = thresh[y:y+h, x:x+w]

    # make square canvas
    size = max(w, h)
    square = np.zeros((size, size), dtype="uint8")

    x_offset = (size - w) // 2
    y_offset = (size - h) // 2
    square[y_offset:y_offset+h, x_offset:x_offset+w] = roi

    digit_img = cv2.resize(square, (28, 28))
    digit_img = digit_img.reshape(1, 28, 28, 1) / 255.0

    pred = model.predict(digit_img, verbose=0)
    digit = np.argmax(pred)

    digits.append((x, str(digit)))

# sort left to right
digits = sorted(digits, key=lambda item: item[0])

number = "".join(d for _, d in digits)

print(number)

In [ ]:
# Sounds good. When you're ready to test, run it with a debug directory so you can inspect the box splits visually:


uv run python elecc_colombia/acta_handwrite_ocr_reader.py
Just update the pdf_path at the bottom of the file to point at one of your downloaded PDFs. The _box0.png, _box1.png, _box2.png debug images will tell you immediately if the splitting is working.